In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DATA_DIR            = Path("../data")
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "mlruns")
EXPERIMENT_NAME     = "csgo-hp-tuning"
N_TRIALS            = 50
N_SPLITS            = 5

In [ ]:
import pandas as pd

X_train = pd.read_csv(DATA_DIR / "X_train.csv", index_col=0)
y_train = pd.read_csv(DATA_DIR / "y_train.csv", index_col=0).squeeze()

print(f"Loaded {len(X_train)} rows, {X_train.shape[1]} features")

In [ ]:
import mlflow
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

def objective(trial: optuna.Trial) -> float:
    C        = trial.suggest_float("C", 1e-3, 100, log=True)
    solver   = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
    max_iter = trial.suggest_int("max_iter", 200, 2000, step=200)

    model  = LogisticRegression(C=C, solver=solver, max_iter=max_iter)
    scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring="roc_auc", n_jobs=-1)
    auc    = float(scores.mean())

    with mlflow.start_run(nested=True):
        mlflow.log_params({"C": C, "solver": solver, "max_iter": max_iter})
        mlflow.log_metric("cv_roc_auc", auc)

    return auc

In [ ]:
with mlflow.start_run(run_name="optuna-search"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best ROC AUC : {study.best_value:.4f}")
print(f"Best params  : {study.best_params}")

In [ ]:
import json

output   = {"params": study.best_params, "cv_roc_auc": study.best_value}
out_path = DATA_DIR / "best_params.json"

with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved to {out_path}")
print("Re-run step 03 to train a final model with these hyperparameters.")